# 05 - Bases analíticas ENIGH

Objetivo: construir dos marts compactos y validados: `mart_persona_2018_2024` y `mart_hogar_2018_2024`. La regla central es evitar merges many-to-many: primero se agregan `ingresos` y `trabajos` a nivel persona.

In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd

try:
    from IPython.display import display, Markdown
except ImportError:
    display = print
    Markdown = lambda text: text


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in [start, *start.parents]:
        if (candidate / "README.md").exists() and (candidate / "data").exists():
            return candidate
    raise FileNotFoundError("No pude localizar la raíz del proyecto.")


ROOT = find_project_root()
RAW = ROOT / "data" / "raw" / "EINGH"
REV3 = ROOT / "data" / "interim" / "revision_3"
REV4 = ROOT / "data" / "interim" / "revision_4"
REV4.mkdir(parents=True, exist_ok=True)

YEARS = [2018, 2020, 2022, 2024]
TABLES = ["viviendas", "hogares", "concentradohogar", "poblacion", "trabajos", "ingresos"]

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 140)

print(f"Proyecto: {ROOT}")
print(f"Datos crudos: {RAW}")

Proyecto: c:\Users\lucia\OneDrive\Escritorio\Fer\inegi-income-modeling
Datos crudos: c:\Users\lucia\OneDrive\Escritorio\Fer\inegi-income-modeling\data\raw\EINGH


In [2]:
PERSON_KEY = ["anio", "folioviv", "foliohog", "numren"]
HOUSE_KEY = ["anio", "folioviv", "foliohog"]
DWELLING_KEY = ["anio", "folioviv"]


def table_path(table):
    return REV3 / f"{table}_decodificada_2018_2024.csv.gz"


def existing_cols(table, candidates):
    header = pd.read_csv(table_path(table), nrows=0)
    return [col for col in candidates if col in header.columns]


def read_table(table, candidates):
    return pd.read_csv(table_path(table), usecols=existing_cols(table, candidates), dtype=str, low_memory=False)


def to_number(series):
    return pd.to_numeric(series, errors="coerce")


def assert_unique(df, key, name):
    dup = int(df.duplicated(key).sum())
    if dup:
        raise ValueError(f"{name} no es único en {key}: {dup} duplicados")
    return dup


def audit_merge(left, right, key, step, mart_key, optional=False):
    before = len(left)
    right_dups = int(right.duplicated(key).sum())
    if right_dups:
        raise ValueError(f"El lado derecho de {step} tiene {right_dups} duplicados en {key}")
    match = left[key].merge(right[key].drop_duplicates().assign(_match=1), on=key, how="left")["_match"].notna()
    matched = int(match.sum())
    merged = left.merge(right, on=key, how="left", validate="m:1")
    after = len(merged)
    if before != after:
        raise ValueError(f"{step}: filas antes={before}, después={after}")
    dup_key = int(merged.duplicated(mart_key).sum())
    if dup_key:
        raise ValueError(f"{step}: el mart quedó con {dup_key} duplicados en {mart_key}")
    return merged, {
        "paso": step,
        "filas_antes": before,
        "filas_despues": after,
        "duplicados_llave_derecha": right_dups,
        "duplicados_llave_mart": dup_key,
        "matches": matched,
        "pct_match": round(matched / before * 100, 4) if before else 100,
        "tipo": "opcional" if optional else "esperado",
    }

## 1. Agregados de ingresos y trabajos a nivel persona

In [3]:
ingresos = read_table("ingresos", PERSON_KEY + ["clave", "clave_desc", "ing_tri"])
ingresos["ing_tri_num"] = to_number(ingresos["ing_tri"]).fillna(0)
clave_num = to_number(ingresos["clave"].astype(str).str.replace("P", "", regex=False))

income_person = (
    ingresos.groupby(PERSON_KEY, dropna=False)
    .agg(
        registros_ingreso=("clave", "size"),
        claves_ingreso_distintas=("clave", "nunique"),
        ingreso_persona_total_registros_tri=("ing_tri_num", "sum"),
    )
    .reset_index()
)

categorias = [
    ("ingreso_persona_laboral_negocio_tri", clave_num.between(1, 22, inclusive="both") | clave_num.between(67, 81, inclusive="both")),
    ("ingreso_persona_rentas_propiedad_tri", clave_num.between(23, 31, inclusive="both")),
    ("ingreso_persona_transferencias_tri", clave_num.between(32, 48, inclusive="both") | clave_num.between(101, 108, inclusive="both")),
    ("ingreso_persona_financiero_capital_tri", clave_num.between(49, 66, inclusive="both")),
]
classified = pd.Series(False, index=ingresos.index)
for name, mask in categorias:
    classified = classified | mask
    agg = ingresos.loc[mask].groupby(PERSON_KEY, dropna=False)["ing_tri_num"].sum().reset_index(name=name)
    income_person = income_person.merge(agg, on=PERSON_KEY, how="left", validate="1:1")
no_classified = ingresos.loc[~classified].groupby(PERSON_KEY, dropna=False)["ing_tri_num"].sum().reset_index(name="ingreso_persona_no_clasificado_tri")
income_person = income_person.merge(no_classified, on=PERSON_KEY, how="left", validate="1:1")
for col in income_person.columns:
    if col.startswith("ingreso_persona_"):
        income_person[col] = income_person[col].fillna(0)
income_person["tiene_registros_ingreso"] = True
income_person.to_csv(REV4 / "ingresos_agregados_persona.csv.gz", index=False, compression="gzip")

trabajos = read_table(
    "trabajos",
    PERSON_KEY + ["id_trabajo", "htrab", "subor", "subor_desc", "indep", "indep_desc", "pago", "pago_desc", "contrato", "contrato_desc", "tiene_suel", "tiene_suel_desc", "tam_emp", "tam_emp_desc"],
)
trabajos["htrab_num"] = to_number(trabajos["htrab"])
trabajos["id_trabajo_num"] = to_number(trabajos["id_trabajo"])
trabajos = trabajos.sort_values(PERSON_KEY + ["id_trabajo_num"], na_position="last")
work_numeric = (
    trabajos.groupby(PERSON_KEY, dropna=False)
    .agg(n_trabajos=("id_trabajo", "size"), horas_trabajos_total=("htrab_num", "sum"), horas_trabajo_principal=("htrab_num", "first"))
    .reset_index()
)
work_principal = trabajos.drop_duplicates(PERSON_KEY, keep="first")[
    PERSON_KEY + ["id_trabajo", "subor", "subor_desc", "indep", "indep_desc", "pago", "pago_desc", "contrato", "contrato_desc", "tiene_suel", "tiene_suel_desc", "tam_emp", "tam_emp_desc"]
].rename(
    columns={
        "id_trabajo": "id_trabajo_principal",
        "subor": "subor_principal",
        "subor_desc": "subor_principal_desc",
        "indep": "indep_principal",
        "indep_desc": "indep_principal_desc",
        "pago": "pago_principal",
        "pago_desc": "pago_principal_desc",
        "contrato": "contrato_principal",
        "contrato_desc": "contrato_principal_desc",
        "tiene_suel": "tiene_suel_principal",
        "tiene_suel_desc": "tiene_suel_principal_desc",
        "tam_emp": "tam_emp_principal",
        "tam_emp_desc": "tam_emp_principal_desc",
    }
)
work_person = work_numeric.merge(work_principal, on=PERSON_KEY, how="left", validate="1:1")
work_person["tiene_trabajo_reportado"] = True
work_person.to_csv(REV4 / "trabajos_agregados_persona.csv.gz", index=False, compression="gzip")

display(income_person.groupby("anio").size().rename("personas_con_ingreso").reset_index())
display(work_person.groupby("anio").agg(personas_con_trabajo=("numren", "size"), max_trabajos=("n_trabajos", "max")).reset_index())

,anio,personas_con_ingreso
0,2018,182434
1,2020,204332
2,2022,205144
3,2024,202365


,anio,personas_con_trabajo,max_trabajos
0,2018,127638,2
1,2020,149387,2
2,2022,150682,2
3,2024,150382,2


## 2. Mart persona

In [ ]:
poblacion = read_table(
    "poblacion",
    PERSON_KEY + 
    ["parentesco", "parentesco_desc", "sexo", 
     "sexo_desc", "edad", "hablaind", "hablaind_desc", 
     "lenguaind", "etnia", "etnia_desc", 
     "alfabetism", "alfabetism_desc", "asis_esc", 
     "asis_esc_desc", "nivel", "nivel_desc", 
     "grado", "nivelaprob", "nivelaprob_desc", 
     "gradoaprob", "residencia", "residencia_desc", 
     "edo_conyug", "edo_conyug_desc", "segsoc", 
     "segsoc_desc", "trabajo_mp", "motivo_aus", 
     "motivo_aus_desc", "act_pnea1", "act_pnea1_desc", 
     "act_pnea2", "act_pnea2_desc", "num_trabaj", 
     "num_trabaj_desc"],
)
assert_unique(poblacion, PERSON_KEY, "poblacion")

concentrado = read_table(
    "concentradohogar",
    HOUSE_KEY + 
    ["ubica_geo", "cve_ent", "entidad", "cve_mun", 
     "municipio", "tam_loc", "tam_loc_desc", "est_socio", 
     "est_socio_desc", "est_dis", "upm", "factor", 
     "clase_hog", "clase_hog_desc", "sexo_jefe", "sexo_jefe_desc", 
     "edad_jefe", "educa_jefe", "educa_jefe_desc", "tot_integ", 
     "hombres", "mujeres", "mayores", "menores", 
     "p12_64", "p65mas", "ocupados", "percep_ing", 
     "perc_ocupa", "ing_cor", "ingtrab", "trabajo", 
     "sueldos", "negocio", "rentas", "transfer", 
     "otros_ing", "gasto_mon"],
)

assert_unique(concentrado, HOUSE_KEY, "concentradohogar")
concentrado = concentrado.rename(
    columns={"factor": "factor_hogar", 
             "ing_cor": "ing_cor_hogar_oficial_tri", 
             "ingtrab": "ingtrab_hogar_oficial_tri", 
             "trabajo": "trabajo_subordinado_hogar_tri", 
             "sueldos": "sueldos_hogar_tri", 
             "negocio": "negocio_hogar_tri", 
             "rentas": "rentas_hogar_tri", 
             "transfer": "transfer_hogar_tri", 
             "otros_ing": "otros_ing_hogar_tri", 
             "gasto_mon": "gasto_mon_hogar_tri"}
             )

hogares = read_table(
    "hogares", HOUSE_KEY + 
    ["telefono", "telefono_desc", "celular", 
     "celular_desc", "tv_paga", "tv_paga_desc", 
     "conex_inte", "conex_inte_desc", "consumo", 
     "consumo_desc", "tarjeta", "tarjeta_desc"]
     )

assert_unique(hogares, HOUSE_KEY, "hogares")
hogares = hogares.rename(columns={col: f"{col}_hogar" for col in hogares.columns if col not in HOUSE_KEY})

viviendas = read_table("viviendas", DWELLING_KEY + 
                       ["tipo_viv", "tipo_viv_desc", "mat_pared", 
                        "mat_pared_desc", "mat_techos", "mat_techos_desc", 
                        "mat_pisos", "mat_pisos_desc", "num_cuarto", 
                        "cuart_dorm", "tenencia", "tenencia_desc", 
                        "dotac_agua", "dotac_agua_desc", "excusado", 
                        "excusado_desc", "drenaje", "drenaje_desc", 
                        "disp_elect", "disp_elect_desc", "combustible", 
                        "combustible_desc"]
                        )

assert_unique(viviendas, DWELLING_KEY, "viviendas")
viviendas = viviendas.rename(columns={col: f"{col}_vivienda" for col in viviendas.columns if col not in DWELLING_KEY})

audit_persona = []
mart_persona = poblacion.copy()
mart_persona, row = audit_merge(mart_persona, income_person, PERSON_KEY, "Agregar ingresos a nivel persona", PERSON_KEY, optional=True); audit_persona.append(row)
mart_persona, row = audit_merge(mart_persona, work_person, PERSON_KEY, "Agregar trabajos a nivel persona", PERSON_KEY, optional=True); audit_persona.append(row)
mart_persona, row = audit_merge(mart_persona, concentrado, HOUSE_KEY, "Agregar hogar/concentrado/geografía", PERSON_KEY); audit_persona.append(row)
mart_persona, row = audit_merge(mart_persona, hogares, HOUSE_KEY, "Agregar variables de hogares", PERSON_KEY); audit_persona.append(row)
mart_persona, row = audit_merge(mart_persona, viviendas, DWELLING_KEY, "Agregar vivienda", PERSON_KEY); audit_persona.append(row)

for col in [c for c in mart_persona.columns if c.startswith("ingreso_persona_")] + ["registros_ingreso", "claves_ingreso_distintas", "n_trabajos", "horas_trabajos_total", "horas_trabajo_principal"]:
    if col in mart_persona.columns:
        mart_persona[col] = to_number(mart_persona[col]).fillna(0)

for col in ["tiene_registros_ingreso", "tiene_trabajo_reportado"]:
    if col in mart_persona.columns:
        mart_persona[col] = mart_persona[col].fillna(False)

for col in ["edad", "factor_hogar", "ing_cor_hogar_oficial_tri", "ingtrab_hogar_oficial_tri", "tot_integ"]:
    mart_persona[col] = to_number(mart_persona[col])

mart_persona["ing_cor_hogar_pc_oficial_tri"] = mart_persona["ing_cor_hogar_oficial_tri"] / mart_persona["tot_integ"].replace(0, np.nan)
mart_persona["ingtrab_hogar_pc_oficial_tri"] = mart_persona["ingtrab_hogar_oficial_tri"] / mart_persona["tot_integ"].replace(0, np.nan)

assert_unique(mart_persona, PERSON_KEY, "mart_persona")

mart_persona.to_csv(REV4 / "mart_persona_2018_2024.csv.gz", 
                    index=False, 
                    compression="gzip")

audit_persona = pd.DataFrame(audit_persona)

audit_persona.to_csv(REV4 / "validacion_mart_persona.csv", 
                     index=False, 
                     encoding="utf-8")

display(audit_persona)
print(mart_persona.shape)

,paso,filas_antes,filas_despues,duplicados_llave_derecha,duplicados_llave_mart,matches,pct_match,tipo
0,Agregar ingresos a nivel persona,1203231,1203231,0,0,794275,66.0118,opcional
1,Agregar trabajos a nivel persona,1203231,1203231,0,0,578089,48.0447,opcional
2,Agregar hogar/concentrado/geografía,1203231,1203231,0,0,1203231,100.0000,esperado
3,Agregar variables de hogares,1203231,1203231,0,0,1203231,100.0000,esperado
4,Agregar vivienda,1203231,1203231,0,0,1203231,100.0000,esperado


(1203231, 138)


## 3. Mart hogar

In [ ]:
pop                 = poblacion.copy()
pop["edad_num"]     = to_number(pop["edad"])
pop["es_menor_18"]  = pop["edad_num"] < 18
pop["es_mayor_65"]  = pop["edad_num"] >= 65
pop["es_jefe"]      = pop["parentesco"].astype(str).str.zfill(3).eq("101")

pop_hogar = (
    pop.groupby(HOUSE_KEY, dropna=False)
    .agg(n_personas_calc=("numren", "size"), n_menores_18_calc=("es_menor_18", "sum"), n_mayores_65_calc=("es_mayor_65", "sum"), edad_promedio_calc=("edad_num", "mean"))
    .reset_index()
)

jefes       = ( 
    pop.loc[pop["es_jefe"], 
            HOUSE_KEY + ["edad_num", "sexo_desc", "nivelaprob_desc"]]
            .rename(columns={"edad_num": "edad_jefe_desde_poblacion", 
                             "sexo_desc": "sexo_jefe_desde_poblacion_desc", 
                             "nivelaprob_desc": "nivelaprob_jefe_desc"})
            .drop_duplicates(HOUSE_KEY)
            )

pop_hogar   = pop_hogar.merge(jefes, 
                              on=HOUSE_KEY, 
                              how="left", 
                              validate="1:1")

pop_hogar.to_csv(REV4 / "poblacion_agregada_hogar.csv.gz", index=False, compression="gzip")

ing_hogar = (
    income_person.groupby(HOUSE_KEY, dropna=False)
    .agg(personas_con_registros_ingreso=("tiene_registros_ingreso", "sum"), 
         ingreso_personas_total_registros_tri=("ingreso_persona_total_registros_tri", "sum"), 
         ingreso_personas_laboral_negocio_tri=("ingreso_persona_laboral_negocio_tri", "sum"), 
         ingreso_personas_transferencias_tri=("ingreso_persona_transferencias_tri", "sum"), 
         ingreso_personas_rentas_propiedad_tri=("ingreso_persona_rentas_propiedad_tri", "sum"), 
         ingreso_personas_financiero_capital_tri=("ingreso_persona_financiero_capital_tri", "sum"))
    .reset_index()
)

audit_hogar = []
mart_hogar = concentrado.copy()

mart_hogar, row = audit_merge(mart_hogar, hogares, HOUSE_KEY, "Agregar variables de hogares", HOUSE_KEY); audit_hogar.append(row)
mart_hogar, row = audit_merge(mart_hogar, viviendas, DWELLING_KEY, "Agregar vivienda", HOUSE_KEY); audit_hogar.append(row)
mart_hogar, row = audit_merge(mart_hogar, pop_hogar, HOUSE_KEY, "Agregar agregados de población", HOUSE_KEY); audit_hogar.append(row)
mart_hogar, row = audit_merge(mart_hogar, ing_hogar, HOUSE_KEY, "Agregar ingresos derivados de ingresos.csv", HOUSE_KEY, optional=True); audit_hogar.append(row)

for col in ["tot_integ", "ocupados", "factor_hogar", "ing_cor_hogar_oficial_tri", "ingtrab_hogar_oficial_tri", "gasto_mon_hogar_tri", "n_personas_calc", "n_menores_18_calc", "n_mayores_65_calc", "personas_con_registros_ingreso"]:
    if col in mart_hogar.columns:
        mart_hogar[col] = to_number(mart_hogar[col])

for col in [c for c in mart_hogar.columns if c.startswith("ingreso_personas_")]:
    mart_hogar[col] = to_number(mart_hogar[col]).fillna(0)

mart_hogar["ing_cor_pc_oficial_tri"] = mart_hogar["ing_cor_hogar_oficial_tri"] / mart_hogar["tot_integ"].replace(0, np.nan)
mart_hogar["ingtrab_pc_oficial_tri"] = mart_hogar["ingtrab_hogar_oficial_tri"] / mart_hogar["tot_integ"].replace(0, np.nan)
mart_hogar["prop_ocupados_oficial"] = mart_hogar["ocupados"] / mart_hogar["tot_integ"].replace(0, np.nan)

assert_unique(mart_hogar, HOUSE_KEY, "mart_hogar")
mart_hogar.to_csv(REV4 / "mart_hogar_2018_2024.csv.gz", index=False, compression="gzip")
audit_hogar = pd.DataFrame(audit_hogar)
audit_hogar.to_csv(REV4 / "validacion_mart_hogar.csv", index=False, encoding="utf-8")

comparacion = mart_hogar[HOUSE_KEY + ["tot_integ", "n_personas_calc", "edad_jefe", "edad_jefe_desde_poblacion"]].copy()

comparacion["diff_tot_integ_vs_poblacion"] = to_number(comparacion["tot_integ"]) - to_number(comparacion["n_personas_calc"])
comparacion["diff_edad_jefe_vs_poblacion"] = to_number(comparacion["edad_jefe"]) - to_number(comparacion["edad_jefe_desde_poblacion"])

comparacion_summary = pd.DataFrame([
    {"comparacion": "tot_integ vs conteo en poblacion", "filas": len(comparacion), "pct_exacto": round((comparacion["diff_tot_integ_vs_poblacion"].abs() <= 0.01).mean() * 100, 4), "max_abs_diff": float(comparacion["diff_tot_integ_vs_poblacion"].abs().max())},
    {"comparacion": "edad_jefe vs jefe identificado en poblacion", "filas": int(comparacion["edad_jefe_desde_poblacion"].notna().sum()), "pct_exacto": round((comparacion["diff_edad_jefe_vs_poblacion"].abs() <= 0.01).mean() * 100, 4), "max_abs_diff": float(comparacion["diff_edad_jefe_vs_poblacion"].abs().max())},
])
comparacion_summary.to_csv(REV4 / "comparacion_variables_hogar.csv", index=False, encoding="utf-8")

display(audit_hogar)
display(comparacion_summary)
print(mart_hogar.shape)

,paso,filas_antes,filas_despues,duplicados_llave_derecha,duplicados_llave_mart,matches,pct_match,tipo
0,Agregar variables de hogares,345169,345169,0,0,345169,100.0000,esperado
1,Agregar vivienda,345169,345169,0,0,345169,100.0000,esperado
2,Agregar agregados de población,345169,345169,0,0,345169,100.0000,esperado
3,Agregar ingresos derivados de ingresos.csv,345169,345169,0,0,344919,99.9276,opcional


,comparacion,filas,pct_exacto,max_abs_diff
0,tot_integ vs conteo en poblacion,345169,99.8734,6.0
1,edad_jefe vs jefe identificado en poblacion,345169,100.0000,0.0


(345169, 90)


## 4. Diccionario y resumen de outputs

In [6]:
dictionary_rows = []
for mart_name, df in [("mart_persona", mart_persona), ("mart_hogar", mart_hogar)]:
    for col in df.columns:
        if col in PERSON_KEY:
            role = "llave persona"
        elif col in HOUSE_KEY:
            role = "llave hogar"
        elif col.endswith("_desc") or col.endswith("_desc_hogar") or col.endswith("_desc_vivienda"):
            role = "etiqueta descriptiva"
        elif col.startswith("ingreso_") or col.startswith("ing_cor") or col.startswith("ingtrab"):
            role = "ingreso"
        elif col in ["factor_hogar", "est_dis", "upm"]:
            role = "diseño muestral"
        elif col in ["entidad", "municipio", "cve_ent", "cve_mun", "tam_loc", "tam_loc_desc", "est_socio", "est_socio_desc"]:
            role = "geografía"
        else:
            role = "variable explicativa"
        dictionary_rows.append({"mart": mart_name, "variable": col, "rol": role})
diccionario_marts = pd.DataFrame(dictionary_rows)
diccionario_marts.to_csv(REV4 / "diccionario_marts.csv", index=False, encoding="utf-8")

summary = {
    "mart_persona": {"archivo": str(REV4 / "mart_persona_2018_2024.csv.gz"), "filas": int(len(mart_persona)), "columnas": int(mart_persona.shape[1]), "llave": PERSON_KEY, "duplicados_llave": int(mart_persona.duplicated(PERSON_KEY).sum())},
    "mart_hogar": {"archivo": str(REV4 / "mart_hogar_2018_2024.csv.gz"), "filas": int(len(mart_hogar)), "columnas": int(mart_hogar.shape[1]), "llave": HOUSE_KEY, "duplicados_llave": int(mart_hogar.duplicated(HOUSE_KEY).sum())},
    "nota_ingresos": "Para hogar se priorizan targets oficiales de concentradohogar; los agregados de ingresos.csv son derivados persona-clave.",
    "nota_factor": "Los marts heredan factor, est_dis y upm desde concentradohogar/viviendas para mantener consistencia 2018-2024.",
}
with open(REV4 / "resumen_revision_4_marts.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2)
display(pd.DataFrame([summary["mart_persona"], summary["mart_hogar"]]))

,archivo,filas,columnas,llave,duplicados_llave
0,c:\Users\lucia\OneDrive\Escritorio\Fer\inegi-income-modeling\data\interim\revision_4\mart_persona_2018_2024.csv.gz,1203231,138,"[anio, folioviv, foliohog, numren]",0
1,c:\Users\lucia\OneDrive\Escritorio\Fer\inegi-income-modeling\data\interim\revision_4\mart_hogar_2018_2024.csv.gz,345169,90,"[anio, folioviv, foliohog]",0
